# Restaurant tipping

Use the sidebar to choose bills by amount and by meal, and the menus in each card to change its plot.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interactive
from IPython.display import display
from statsmodels.nonparametric.smoothers_lowess import lowess

In [2]:
TIPS = sns.load_dataset('tips')
TIPS['percent'] = TIPS.tip / TIPS.total_bill
bill_rng = (TIPS.total_bill.min(), TIPS.total_bill.max())

In [3]:
bill = widgets.FloatRangeSlider(value=bill_rng, min=bill_rng[0], max=bill_rng[1], step=0.01,
                                readout_format='.2f', continuous_update=False)
lunch = widgets.Checkbox(value=True, description='Lunch', indent=False)
dinner = widgets.Checkbox(value=True, description='Dinner', indent=False)
reset_button = widgets.Button(description='Reset filter')

sidebar = widgets.VBox([
    widgets.Label('Bill amount ($)'), bill,
    widgets.Label('Food service'), widgets.HBox([lunch, dinner]),
    reset_button,
], layout=widgets.Layout(width='320px'))

def reset(button):
    bill.value = bill_rng
    lunch.value = True
    dinner.value = True

reset_button.on_click(reset)

In [4]:
def filter_tips(bill, lunch, dinner):
    times = []
    if lunch:
        times.append('Lunch')
    if dinner:
        times.append('Dinner')
    keep = TIPS.total_bill.between(bill[0], bill[1]) & TIPS.time.isin(times)
    return TIPS[keep]

def value_box(title, value):
    html = (f'<div style="padding: 4px 16px;">'
            f'<div style="color: gray;">{title}</div>'
            f'<div style="font-size: 2em; font-weight: bold;">{value}</div>'
            f'</div>')
    return widgets.HTML(html, layout=widgets.Layout(border='1px solid lightgray', width='200px'))

def card(title, children):
    header = widgets.HTML(f'<b>{title}</b>')
    return widgets.VBox([header] + children,
                        layout=widgets.Layout(border='1px solid lightgray', padding='8px', margin='4px'))

In [5]:
def show_values(bill, lunch, dinner):
    df = filter_tips(bill, lunch, dinner)
    if len(df) == 0:
        avg_tip = '—'
        avg_bill = '—'
    else:
        avg_tip = f'{df.percent.mean():.1%}'
        avg_bill = f'${df.total_bill.mean():.2f}'
    display(widgets.HBox([
        value_box('Total tippers', len(df)),
        value_box('Average tip', avg_tip),
        value_box('Average bill', avg_bill),
    ]))

def show_table(bill, lunch, dinner):
    df = filter_tips(bill, lunch, dinner).drop(columns='percent')
    with pd.option_context('display.max_rows', None):
        display(df)

color = widgets.Dropdown(options=['none', 'sex', 'smoker', 'day', 'time'], value='none', description='Color')

def show_scatter(bill, lunch, dinner, color):
    df = filter_tips(bill, lunch, dinner)
    if len(df) < 3:
        print('Too few tippers to plot.')
        return
    if color == 'none':
        ax = df.plot.scatter(x='total_bill', y='tip', figsize=(5, 3.5))
    else:
        ax = df.plot.scatter(x='total_bill', y='tip', c=color, cmap='tab10', figsize=(5, 3.5))
    trend = pd.DataFrame(lowess(df.tip, df.total_bill), columns=['total_bill', 'tip'])
    trend.plot(x='total_bill', y='tip', ax=ax, color='black', legend=False)
    plt.show()

split = widgets.Dropdown(options=['sex', 'smoker', 'day', 'time'], value='day', description='Split by')

def show_percents(bill, lunch, dinner, split):
    by_group = (filter_tips(bill, lunch, dinner)
        .pivot(columns=split, values='percent')
        .dropna(axis=1, thresh=2))
    if by_group.shape[1] == 0:
        print('Too few tippers to plot.')
        return
    by_group.plot.density(subplots=True, sharex=True, xlim=(0, 0.75), figsize=(10, 5))
    plt.show()

In [6]:
values = interactive(show_values, bill=bill, lunch=lunch, dinner=dinner)
table = interactive(show_table, bill=bill, lunch=lunch, dinner=dinner)
scatter = interactive(show_scatter, bill=bill, lunch=lunch, dinner=dinner, color=color)
percents = interactive(show_percents, bill=bill, lunch=lunch, dinner=dinner, split=split)

table.children[-1].layout = widgets.Layout(height='300px', overflow='auto')

In [7]:
middle = widgets.HBox([
    card('Tips data', [table.children[-1]]),
    card('Total bill vs tip', [color, scatter.children[-1]]),
])
bottom = card('Tip percentages', [split, percents.children[-1]])
main = widgets.VBox([values.children[-1], middle, bottom])

widgets.HBox([sidebar, main])